In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot
from scipy.stats import binom
print(np.__version__)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
#import seaborn

import os
import sys
if '../lib/' not in sys.path:
    sys.path.append('../lib/')
import dio3
import multi_fgl_utils as util
import plot_util as putil

1.24.3


In [2]:
if 1:
    import importlib
    importlib.reload(util);


In [3]:
df2tex = util.df2tex

In [4]:
script_fn = 'fermi_multiclass/notebooks/pcat_summary.ipynb'
fig_meta = {'Author': 'Dmitry Malyshev', 'Subject': script_fn}

sample_std = False
# select the catalog and the case
cat_name = '4FGL-DR4' # '4FGL_DR3'
folder_out = None
if cat_name == '4FGL-DR4':
    #baseline_case = '4FGL-DR4_GMM_RF'
    #case = '4FGL-DR4_GMM_RF'
    #case = '4FGL-DR4_GMM_RF_no_bcu_spp'
    #case = '4FGL-DR4_GMM_RF_no_bcu'
    #case = '4FGL-DR4_GMM_RF_no_coord'
    #case = '4FGL-DR4_GMM_RF_no_coord_w'
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp'
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp_w'
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu'
    #case = '4FGL-DR4_GMM_PDF_pshift_nGauss1_npar4'
    #case = '4FGL-DR4_GMM_PDF_no_coord_pshift_nGauss1_npar4'
    #case = '4FGL-DR4_GMM_PDF_no_coord_no_bcu_pshift_nGauss1_npar4'
    #case = '4FGL-DR4_GMM_PDF_no_bcu_pshift_nGauss1_npar4'
    #case = '4FGL-DR4_GMM_PDF_no_bcu_spp_pshift_nGauss1_npar4'
    #case = '4FGL-DR4_GMM_PDF_no_coord_no_bcu_spp_pshift_nGauss1_npar4'
    #case = '4FGL-DR4_GMM_PDF_no_coord_no_bcu_spp_pshift_nGauss0_npar4'
    
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp_nmin50'
    baseline_case = '4FGL-DR4_2class_RF' # '4FGL-DR4_GMM_RF', '4FGL-DR4_2class_RF'
    case = '4FGL-DR4_2class_PDF_no_coord_no_bcu_spp_pshift_nGauss1_npar4'
    #case = '4FGL-DR4_2class_pulsars_AGNs'
    #case = '4FGL-DR4_GMM_RF'
    #baseline_case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp'
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp'
    #baseline_case = '4FGL-DR4_GMM_RF_no_coord'
    #case = '4FGL-DR4_GMM_RF_no_coord'
    #case = '4FGL-DR4_GMM_RF_no_coord_w'
if cat_name == '4FGL-DR3':
    #baseline_case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp'
    #case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp_nmin50' # '4FGL-DR4_GMM_RF_no_prune'
    baseline_case = '4FGL-DR3_3class_RF'
    #case = '4FGL-DR3_3class_RF'
    #case = '4FGL-DR3_3class_spec_features_RF'
    baseline_case = '4FGL-DR3_3class_Toby'
    case = '4FGL-DR3_3class_Toby' # '4FGL_DR4_2class', '4FGL_DR4_RF_RF'


baseline = (case == baseline_case)
root_folder = '../'
config_folder = root_folder + 'config/'

# generic inputs
cdict = util.get_config_dict(cat_name=cat_name, 
                             baseline_case=baseline_case, 
                             case=case, 
                             folder=config_folder)

cdict['root_folder'] = root_folder
#folder_in = cdict['root_folder'] + cdict.get('folder_in', 'data_in/')
#folder_out = cdict['root_folder'] + cdict.get('folder_out', 'data_out/')
#folder = folder_out
Eind = int(cdict.get('Eind', 1000))
class_name = cdict.get('class_name', 'CLASS1')

assoc_sources = cdict.get('assoc_sources', 'all_assoc')
sep_model = cdict.get('sep_model', 'GMM')
#sep_model = 'RF' # 'GMM', 'RF'
nmin = cdict.get('nmin')
if nmin is not None: 
    nmin = int(nmin)
max_depth = int(cdict.get('max_depth', 5))
feature_mode = cdict.get('features', 'all_features')

alg = cdict.get('class_alg', 'RF')
train_weighting = cdict.get('train_weighting')
test_weighting = cdict.get('test_weighting')
weighted = (train_weighting is not None) #or (test_weighting is not None)
#oversample_factor = 1 # used for 'resampled' training and testing
max_sample = int(cdict.get('max_sample', 7))
test_frac = float(cdict.get('test_frac', 0.3))
nav = int(cdict.get('nav', 10))
add_pshift = int(cdict.get('prior_shift', 0))

tree_alg = cdict.get('sep_model', 'GMM')
#tree_alg = 'RF' # 'stat', 'GMM', 'RF', '4class', '2class', 'RF_aakash'


# input data
tree_cat_fn = util.get_tree_cat_fn(cdict)
tree_cat_dict = dio3.loaddict(tree_cat_fn)
if util.exist_internal(tree_cat_dict):
    util.remove_internal(tree_cat_dict)
cat_tree = list(tree_cat_dict.keys())
nclasses = len(tree_cat_dict)
max_tree_cat_dict = util.tree2max_tree(tree_cat_dict)
dec_tree_cat_dict = util.max_tree2dec_tree(max_tree_cat_dict)
nodes = [str(key) for key in dec_tree_cat_dict.keys()]
if add_pshift:
    if int(cdict.get('nGauss', 0)):
        nclasses += 1
        nodes.append('Gauss')
        


xdf_fn = util.get_xdf_fn(cdict)
xdf = pd.read_csv(xdf_fn, index_col=0)
ns = len(xdf)

all_features = [name for name in xdf.columns if name.find('CLASS') == -1]
if feature_mode == 'all_features':
    features = all_features
elif feature_mode == 'no_coord_features':
    features = [feat for feat in all_features if feat.find('GL') == -1]


nsrc_dict = util.get_nsrc_dict(xdf, class_column=class_name, classes='associated')
nsrc_df = util.nsrc_dict2df(nsrc_dict)
#assoc_classes = list(nsrc_dict.keys())
assoc_classes = []
for cls in dec_tree_cat_dict.values():
    assoc_classes.extend(cls)
all_classes = list(set(xdf[class_name]))
unas_classes = [cl for cl in all_classes if cl not in assoc_classes]
unas_order = ['unas', 'unk', 'bcu', 'spp']
ucls = []
for cl in unas_order:
    if cl in unas_classes:
        unas_classes.remove(cl)
        ucls.append(cl)
ucls.extend(unas_classes)
unas_classes = ucls
print(unas_classes)


folder_out = '%sresults/%s_probabilistic_catalogs/' % (root_folder, util.get_cat_descr(cdict, add_pshift=0))
fn = '%s%s_%s' % (folder_out, util.get_cat_descr(cdict, add_pshift=add_pshift), 'prob_cat.csv')
folder_gu = '../../Fermi_SGU/GU_git/data_in/ML_catalogs/'
fn_sum_gu = '%s%s_%s' % (folder_gu, util.get_cat_descr(cdict, add_pshift=add_pshift), 'summary.csv')
print(fn)
print(fn_sum_gu)

#folder_out = '../results/%s_probabilistic_classification_weighted/' % cat_name
#fn = folder_out + util.get_cat_descr(cdict)
summary_fn = fn.replace('prob_cat.csv', 'summary.csv')
if sample_std:
    all_folder_out = fn.replace('.csv', '_all_pred/')

unas_key = ''
for cl in unas_classes:
    unas_key += '%s,' % cl
unas_key = unas_key[:-1]
print(unas_key)

cat_df = pd.read_csv(fn, index_col=0)
if case.find('Toby') > -1:
    cat_df['CLASS1'] = ['%s' %s.split('-')[0] for s in cat_df['source_type']]
    unas_mask = cat_df['CLASS1'] == 'unid'
    cat_df['CLASS1'][unas_mask] = 'unas'
    algs = [cdict['class_alg']]
    alg_len = 2
    for node in nodes:
        for alg in algs:
            cl = '%s_%s' % (node, alg)
            cat_df[cl + '_std'] = 0.01 * cat_df[cl] # hack/placeholder

    nmax = len(xdf.index[0])
    xdf.index = [x[:17] for x in xdf.index]
    xdf = xdf.loc[cat_df.index]
elif int(cdict.get('prior_shift', 0)):
    algs = [cdict['class_alg']]
    alg_len = 3
else:
    algs = ['RF', 'NN']
    alg_len = 2
unas_mask = (cat_df['CLASS1'] == 'unas') | (cat_df['CLASS1'] == 'unk')
ns = len(cat_df)

['unas', 'unk', 'bcu', 'spp']
../results/4FGL-DR4_v34_2classes_2class_gal_egal_no_bcu_spp_no_coord_features_probabilistic_catalogs/4FGL-DR4_v34_2classes_2class_gal_egal_no_bcu_spp_no_coord_features_unas_unk_prior_shift_npar4_1Gauss_prob_cat.csv
../../Fermi_SGU/GU_git/data_in/ML_catalogs/4FGL-DR4_v34_2classes_2class_gal_egal_no_bcu_spp_no_coord_features_unas_unk_prior_shift_npar4_1Gauss_summary.csv
unas,unk,bcu,spp


In [5]:
add_pshift

1

In [6]:
algs, nodes

(['PDF'], ['1', '2', 'Gauss'])

In [7]:
cat_df.columns

Index(['log10(Energy_Flux100)', 'log10(LP_EPeak)', 'LP_beta', 'GLON', 'GLAT',
       'Signif_Avg', 'CLASS1', '1_PDF', '2_PDF', 'Gauss_PDF'],
      dtype='object')

In [8]:
pcols = [cl for cl in cat_df.columns if cl[-alg_len:] in algs]
pcols

['1_PDF', '2_PDF', 'Gauss_PDF']

In [9]:
# sources with maximal probabilities
src = []
for node in nodes:
    dfc = 0.
    for alg in algs:
        cl = '%s_%s' % (node, alg)
        dfc += cat_df[cl][unas_mask] / len(algs)
    dfc = dfc.sort_values(ascending=False)
    src.extend(dfc.index[:1])

In [10]:
len(src), algs

(3, ['PDF'])

In [11]:
cols = ['Source_Name', 'Group']
for alg in algs:
    cols.append('%s prob' % alg)
df_out = pd.DataFrame(columns=cols, index=range(1, len(src) + 1))
i = 0
for node in nodes:
    print(node)
    if node == 'Gauss':
        df_out.loc[i + 1, 'Group'] = node
    else:
        df_out.loc[i + 1, 'Group'] = util.cl_list2str(dec_tree_cat_dict[int(node)])
    df_out.loc[i, 'Source_Name'] = src[i]
    for alg in algs:
        df_out.loc[i + 1, '%s prob' % alg] = cat_df['%s_%s' % (node, alg)][src[i]]
    i += 1

1
2
Gauss


In [12]:
#df_out.index = 

In [13]:
df_out

,Source_Name,Group,PDF prob
1,4FGL J1801.1-2626,"bll, fsrq, rdg, agn, ssrq, css, nlsy1, sey",0.999264
2,4FGL J0614.1+1628c,"psr, msp, gc, pwn, snr, glc, sfr, hmb, lmb, bi...",0.99999
3,NaN,Gauss,0.994003
0,4FGL J0859.8+0053,NaN,NaN


In [14]:
print(df2tex(df_out, fmt='%.3f', index=False))

\begin{tabular}{lll}
\hline
Source_Name & Group & PDF prob \\
\hline
4FGL J1801.1-2626 & bll, fsrq, rdg, agn, ssrq, css, nlsy1, sey & 0.999 \\
4FGL J0614.1+1628c & psr, msp, gc, pwn, snr, glc, sfr, hmb, lmb, bin, nov, sbg, gal & 1.000 \\
NaN & Gauss & 0.994 \\
4FGL J0859.8+0053 & NaN & NaN \\
\hline
\end{tabular}



In [15]:
print(df2tex(cat_df[pcols].loc[src], fmt='%.3f', index=True, index_name='Source\_Name'))

\begin{tabular}{lrrr}
\hline
 & 1_PDF & 2_PDF & Gauss_PDF \\
\hline
4FGL J0859.8+0053 & 0.999 & 0.001 & 0.000 \\
4FGL J1801.1-2626 & 0.000 & 1.000 & 0.000 \\
4FGL J0614.1+1628c & 0.000 & 0.006 & 0.994 \\
\hline
\end{tabular}



In [16]:
max_tree_cat_dict

{'00': ['bll', 'fsrq', 'rdg', 'agn', 'ssrq', 'css', 'nlsy1', 'sey'],
 '01': ['psr',
  'msp',
  'gc',
  'pwn',
  'snr',
  'glc',
  'sfr',
  'hmb',
  'lmb',
  'bin',
  'nov',
  'sbg',
  'gal']}

In [17]:
# summary of predictions


clms = ['Node', 'Physical classes', 'N assoc']
for alg in algs:
    clms.append('%s assoc' % alg)
for alg in algs:
    clms.append('%s %s' % (alg, unas_key))
sdf = pd.DataFrame(columns=clms, index=range(1, nclasses + 1))
for i, node in enumerate(nodes):
    if node == 'Gauss':
        sdf.loc[i + 1, 'Physical classes'] = node
        sdf.loc[i + 1, 'Node'] = ''
    else:
        cls = dec_tree_cat_dict[int(node)]
        sdf.loc[i + 1, 'Physical classes'] = util.cl_list2str(cls)
        bnode = list(max_tree_cat_dict.keys())[i]
        sdf.loc[i + 1, 'Node'] = bnode
#sdf.index = range(1, len(sdf) + 1)

if 0:
    #sdf.index.name = 'Index'
    out_clms = ['Node', 'Classes']
    print(sdf[out_clms])
    tst = util.df2tex(sdf[out_clms], fmt='%.1f', index=True, index_name='')
    print(tst)
    
#clf_mask = (cat_df['CLASS1'] != 'unas') & (cat_df['CLASS1'] != 'unk')
clf_mask = np.zeros(ns, dtype=bool)
for key, cls in dec_tree_cat_dict.items():
    print(key, cls)
    cls_mask = util.get_cl_mask(cls, cat_df, cname='CLASS1')
    #ys[cls_mask] = key
    clf_mask[cls_mask] = True

for i, node in enumerate(nodes):
    if node == 'Gauss':
        sdf.loc[i + 1, 'N assoc'] = 0.
        sdf.loc[i + 1, 'Node'] = ''
    else:
        cls = dec_tree_cat_dict[int(node)]
        bnode = list(max_tree_cat_dict.keys())[i]
        sdf.loc[i + 1, 'Node'] = bnode
        nsrc = np.sum([nsrc_dict[cl] for cl in cls])
        sdf.loc[i + 1, 'N assoc'] = nsrc
    for alg in algs:
        nas = np.sum(cat_df[clf_mask]['%s_%s' % (node, alg)])
        nunas = np.sum(cat_df[~clf_mask]['%s_%s' % (node, alg)])
        std_key = '%s_%s_std' % (node, alg)
        if std_key in cat_df:
            nas_std = cat_df[clf_mask][std_key]
            nas_std = np.sqrt(np.sum(nas_std**2))
            nunas_std = cat_df[~clf_mask]['%s_%s_std' % (node, alg)]
            nunas_std = np.sqrt(np.sum(nunas_std**2))
            sdf.loc[i + 1, '%s assoc' % alg] = '$%.1f\pm %.1f$' % (nas, nas_std)
            sdf.loc[i + 1, '%s %s' % (alg, unas_key)] = '$%.1f\pm %.1f$' % (nunas, nunas_std)
        else:
            sdf.loc[i + 1, '%s assoc' % alg] = '$%.1f$' % (nas)
            sdf.loc[i + 1, '%s %s' % (alg, unas_key)] = '$%.1f$' % (nunas)

if sample_std:
    sdf_sample = sdf.copy()
    #fn = all_folder_out + 'ntest.npy'
    #ntest = np.load(fn)
    nunas_dict = {}
    nas_dict = {}
    for alg in algs:
        for i, node in enumerate(nodes):
            clm = '%s_%s' % (node, alg)
            nunas_dict[clm] = []
            nas_dict[clm] = []
    for alg in algs:
        for fn in os.listdir(all_folder_out):
            if fn.startswith(alg):
                ps = np.load(all_folder_out + fn).T
                for i, node in enumerate(nodes):
                    nunas = np.sum(ps[i][~clf_mask])
                    clm = '%s_%s' % (node, alg)
                    nunas_dict[clm].append(nunas)
                    nas = np.sum(ps[i][clf_mask]) / test_frac
                    nas_dict[clm].append(nas)

    for alg in algs:
        for i, node in enumerate(nodes):
            clm = '%s_%s' % (node, alg)
            nunas = np.mean(nunas_dict[clm])
            nunas_std = np.std(nunas_dict[clm])
            sdf_sample.loc[i + 1, '%s %s' % (alg, unas_key)] = '$%.1f\pm %.1f$' % (nunas, nunas_std)
            nas = np.mean(nas_dict[clm])
            nas_std = np.std(nas_dict[clm])
            sdf_sample.loc[i + 1, '%s assoc' % alg] = '$%.1f\pm %.1f$' % (nas, nas_std)
    sdf0 = sdf.copy()
    sdf = sdf_sample
#print(sdf)

1 ['bll', 'fsrq', 'rdg', 'agn', 'ssrq', 'css', 'nlsy1', 'sey']
2 ['psr', 'msp', 'gc', 'pwn', 'snr', 'glc', 'sfr', 'hmb', 'lmb', 'bin', 'nov', 'sbg', 'gal']


In [18]:
#sdf_sample

In [19]:
sdf

,Node,Physical classes,N assoc,PDF assoc,"PDF unas,unk,bcu,spp"
1,00,"bll, fsrq, rdg, agn, ssrq, css, nlsy1, sey",2390,$1407.2$,$1499.3$
2,01,"psr, msp, gc, pwn, snr, glc, sfr, hmb, lmb, bi...",477,$470.1$,$794.2$
3,,Gauss,0.0,$433.7$,$1254.5$


In [20]:
#np.mean(cat_df[~clf_mask]['%s_%s_std' % (node, alg)])

In [21]:
if 0:
    max_classes = [util.get_max_class(dec_tree_cat_dict[int(key)], nsrc_df) + '+' for key in nodes]
else:
    max_classes = []
    for key in nodes:
        if key == 'Gauss':
            max_classes.append(key)
        else:
            clabel = util.get_max_class(dec_tree_cat_dict[int(key)], nsrc_df)
            if len(dec_tree_cat_dict[int(key)]) > 1 or True:
                clabel += '+'
            max_classes.append(clabel)
    sdf.insert(0, 'Class label', max_classes)
print(max_classes)

['bll+', 'msp+', 'Gauss']


In [22]:
# class definition columns
cdef_clms = ['Class label', 'Physical classes', 'N assoc']
txt = util.df2tex(sdf[cdef_clms], fmt='%.1f', index=False, index_name='')
txt = txt.replace('N assoc', 'Number of assoc. sources')
print(txt)

\begin{tabular}{lll}
\hline
Class label & Physical classes & Number of assoc. sources \\
\hline
bll+ & bll, fsrq, rdg, agn, ssrq, css, nlsy1, sey & 2390 \\
msp+ & psr, msp, gc, pwn, snr, glc, sfr, hmb, lmb, bin, nov, sbg, gal & 477 \\
Gauss & Gauss & 0.0 \\
\hline
\end{tabular}



In [23]:
#tst = util.df2tex(sdf, fmt='%.1f', index=True, index_name='')
out_clms = list(sdf.columns)
out_clms.remove('Node')
#out_clms.remove('Physical classes')
tst = util.df2tex(sdf[out_clms], fmt='%s', index=True, index_name='')
#tst = util.df2tex(sdf, fmt='%s', index=True, index_name='')
tst = tst.replace('_', '\_')
tst = tst.replace('\$', '$')
tst = tst.replace('\\textbackslash ', '\\')
while '  ' in tst:
    tst = tst.replace('  ', ' ')
if weighted:
    print('Weighted\n')
else:
    print('Unweighted\n')
print(tst)

Unweighted

\begin{tabular}{llllll}
\hline
 & Class label & Physical classes & N assoc & PDF assoc & PDF unas,unk,bcu,spp \\
\hline
1 & bll+ & bll, fsrq, rdg, agn, ssrq, css, nlsy1, sey & 2390 & $1407.2$ & $1499.3$ \\
2 & msp+ & psr, msp, gc, pwn, snr, glc, sfr, hmb, lmb, bin, nov, sbg, gal & 477 & $470.1$ & $794.2$ \\
3 & Gauss & Gauss & 0.0 & $433.7$ & $1254.5$ \\
\hline
\end{tabular}



In [24]:
algs

['PDF']

In [25]:
for i, node in enumerate(nodes):
    for alg in algs:
        st = sdf['%s assoc' % alg].iloc[i]
        st = st.replace('$', '')
        st = st.replace('\pm', ' +- ')
        sdf.loc[i + 1, '%s assoc' % alg] = st
        st = sdf['%s %s' % (alg, unas_key)].iloc[i]
        st = st.replace('$', '')
        st = st.replace('\pm', ' +- ')
        sdf.loc[i + 1, '%s %s' % (alg, unas_key)] = st
        
save_res = True
copy2gu = 0
if save_res:
    print('Save predictions summary to file:')
    print(summary_fn)
    sdf.to_csv(summary_fn, index_label='Group')
    if copy2gu:
        cmd = 'cp %s %s' % (summary_fn, fn_sum_gu)
        print(cmd)
        os.system(cmd)


Save predictions summary to file:
../results/4FGL-DR4_v34_2classes_2class_gal_egal_no_bcu_spp_no_coord_features_probabilistic_catalogs/4FGL-DR4_v34_2classes_2class_gal_egal_no_bcu_spp_no_coord_features_unas_unk_prior_shift_npar4_1Gauss_summary.csv


In [26]:
sdf

,Class label,Node,Physical classes,N assoc,PDF assoc,"PDF unas,unk,bcu,spp"
1,bll+,00,"bll, fsrq, rdg, agn, ssrq, css, nlsy1, sey",2390,1407.2,1499.3
2,msp+,01,"psr, msp, gc, pwn, snr, glc, sfr, hmb, lmb, bi...",477,470.1,794.2
3,Gauss,,Gauss,0.0,433.7,1254.5


In [27]:
alg = algs[0]
if case.find('no_bcu_spp') > -1:
    unas_clm = '%s unas,unk,bcu,spp' % alg
elif case.find('no_bcu') > -1:
    unas_clm = '%s unas,unk,bcu' % alg
else:
    unas_clm = '%s unas,unk' % alg
few_columns = ['Class label', 'N assoc', unas_clm]
sdf_mini = sdf[few_columns].sort_values(by=['N assoc'], axis=0, ascending=True)
nunas_loc = np.array([float(x.split(' +- ')[0]) for x in sdf_mini[unas_clm]])
sdf_mini['frac increase'] = nunas_loc / (sdf_mini['N assoc'] + 0.1)
print(cat_name)
sdf_mini

4FGL-DR4


,Class label,N assoc,"PDF unas,unk,bcu,spp",frac increase
3,Gauss,0.0,1254.5,12545.0
2,msp+,477,794.2,1.664641
1,bll+,2390,1499.3,0.627296


In [28]:
nunas_loc

array([1254.5,  794.2, 1499.3])

In [29]:
# predictions for unas classes 
print_fraction = True
if 1:
    #ualgs = ['RF', 'NN']
    ualgs = algs
    uclms = ['Class label']
    for ucl in unas_classes:
        for alg in ualgs:
            uclms.append('%s %s' % (alg, ucl))
    usdf = pd.DataFrame(columns=uclms, index=range(1, nclasses + 1))
    usdf['Class label'] = sdf['Class label']
    for ucl in unas_classes:
        ucls_mask = util.get_cl_mask([ucl], xdf, cname=class_name)
        for alg in ualgs:
            for i, node in enumerate(nodes):
                nunas = np.sum(cat_df.loc[ucls_mask, '%s_%s' % (node, alg)])
                if print_fraction:
                    usdf.loc[i + 1, '%s %s' % (alg, ucl)] = '%.2f' % (nunas / sum(ucls_mask))
                else:
                    nunas_std = cat_df.loc[ucls_mask, '%s_%s_std' % (node, alg)]
                    nunas_std = np.sqrt(np.sum(nunas_std**2))
                    usdf.loc[i + 1, '%s %s' % (alg, ucl)] = '$%.1f\pm %.1f$' % (nunas, nunas_std)
                    
if sample_std:
    usdf_sample = usdf.copy()
    nunas_dict = {}
    for alg in algs:
        nunas_dict[alg] = {}
        for i, node in enumerate(nodes):
            nunas_dict[alg][node] = {}
            for ucl in unas_classes:
                nunas_dict[alg][node][ucl] = []
    for alg in algs:
        for fn in os.listdir(all_folder_out):
            if fn.startswith(alg):
                ps = np.load(all_folder_out + fn).T
                for ucl in unas_classes:
                    ucls_mask = util.get_cl_mask([ucl], xdf, cname=class_name)
                    for i, node in enumerate(nodes):
                        nunas = np.sum(ps[i][ucls_mask])
                        nunas_dict[alg][node][ucl].append(nunas)

    for alg in algs:
        for i, node in enumerate(nodes):
            for ucl in unas_classes:
                nunas = np.mean(nunas_dict[alg][node][ucl])
                nunas_std = np.std(nunas_dict[alg][node][ucl])
                usdf_sample.loc[i + 1, '%s %s' % (alg, ucl)] = '$%.1f\pm %.1f$' % (nunas, nunas_std)
                
    usdf0 = usdf.copy()
    usdf = usdf_sample
#print(sdf)

In [30]:
txt = util.df2tex(usdf, fmt='%.1f', index=False, index_name='')
print(txt)

\begin{tabular}{lllll}
\hline
Class label & PDF unas & PDF unk & PDF bcu & PDF spp \\
\hline
bll+ & 0.29 & 0.25 & 0.46 & 0.19 \\
msp+ & 0.22 & 0.17 & 0.13 & 0.20 \\
Gauss & 0.34 & 0.40 & 0.18 & 0.49 \\
\hline
\end{tabular}



In [31]:
print(cat_name)
usdf.sort_values(by=['Class label'], axis=0, ascending=True)

4FGL-DR4


,Class label,PDF unas,PDF unk,PDF bcu,PDF spp
3,Gauss,0.34,0.40,0.18,0.49
1,bll+,0.29,0.25,0.46,0.19
2,msp+,0.22,0.17,0.13,0.20


In [32]:
# calculate the mixing matrix
# input data

ys = np.zeros(ns, dtype=int)
clf_mask = np.zeros(ns, dtype=bool)
cls_masks_all = []
for key, cls in dec_tree_cat_dict.items():
    cls_mask = util.get_cl_mask(cls, xdf, cname=class_name)
    cls_masks_all.append(cls_mask)
    ys[cls_mask] = int(key)
    clf_mask[cls_mask] = True
cls_masks_all = np.array(cls_masks_all)

y_true = ys[clf_mask]
ncl = sum(clf_mask)
labels = max_classes
if case == '4FGL-DR4_2class_pulsars_AGNs':
    labels = ['AGNs', 'pulsars']

IndexError: boolean index did not match indexed array along dimension 0; dimension is 5859 but corresponding boolean dimension is 7191

In [ ]:
cm_dict = {}
#normalize = 'true' #'pred' # None, 'true', 'pred'
normalize = None
norm_dict = {'true': 'true_norm', 'pred': 'pred_norm', None: 'counts'}
for alg in ['RF', 'NN']:
    probs = np.array([cat_df[clf_mask]['%s_%s' % (node, alg)] for node in nodes])
    cm_dict[alg] = {}

    for weighted_test in [0, 1]:
        if weighted_test:
            sample_weight = np.array(cat_df['sample_weight'][clf_mask])
        else:
            sample_weight = np.ones(sum(clf_mask)) # None
        max_prob_prediction = 1
        if max_prob_prediction:
            y_pred = np.argmax(probs, axis=0) + 1
            cm = confusion_matrix(y_true, y_pred, sample_weight=sample_weight, 
                                  normalize=normalize)
        else:
            # calculate prediction based on the sum of probabilities
            cm = np.zeros((len(nodes), len(nodes)))
            for i in range(len(nodes)):
                mask = y_true == i + 1
                cm[i] = np.sum((probs[:, mask]*sample_weight[mask]), axis=1)
            if normalize == 'pred':
                norm = np.sum(cm, axis=0)
                cm /= norm
            elif normalize == 'true':
                norm = np.sum(cm, axis=1)
                cm = (cm.T / norm).T 

        if normalize == None:
            cm = np.around(cm, decimals=2)
        else:
            cm = np.around(cm, decimals=3)
        cm_dict[alg][weighted_test] = cm

In [ ]:
plots_folder = '../plots/comparisons/'
paper_folder = '/Users/Dmitry/Work/Fermi/fermi_multiclass_papers/3_cov_shift_paper\
/cov_shift_paper_main/plots/confusion_matrix/'
if not os.path.isdir(paper_folder):
    os.makedirs(paper_folder)
save_plots = 0
save_png = 1
copy2tex = 0


single_plot_pars = {'bottom':0.12, 'left':0.15, 'right':0.99, 'top': 0.92,
                  #'axes.titlesize':18,'font.size':13
                 }

#putil.setup_figure_pars()
putil.setup_figure_pars(pars_dict=single_plot_pars)

for alg in ['RF', 'NN']:
    for weighted_test in [0, 1]:
        cm = cm_dict[alg][weighted_test]
        title = '%s, ' % alg
        alg_str = '%s_%s' % (alg, norm_dict[normalize])
        if weighted and weighted_test:
            title += 'weighted training and testing'
            fn_str = '_wtraintest_%s' % (alg_str)
        elif weighted:
            title += 'weighted training, unweighted testing'
            fn_str = '_wtrain_%s' % alg_str
        elif weighted_test:
            title += 'unweighted training, weighted testing'
            fn_str = '_wtest_%s' % alg_str
        else:
            title += 'unweighted training and testing'
            fn_str = '_%s' % alg_str

        if 1:
            fig, ax = pyplot.subplots()
            if normalize is None:
                seaborn.heatmap(cm, annot=True, cmap='viridis', fmt='.1f')
            else:
                seaborn.heatmap(cm, annot=True, cmap='viridis', fmt='.2f')
            ax.set_xticklabels(labels)
            ax.set_yticklabels(labels, rotation=0)
            pyplot.xlabel('Predicted')
            pyplot.ylabel('True')
        else:
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
            disp.plot(colorbar=True)
        if case != '4FGL-DR4_2class_pulsars_AGNs':
            pyplot.title(title)


        if save_plots:
            fn = '%s/confusion_matrix_%s%s.pdf' % (plots_folder, cat_name, fn_str)
            print('save figure to file: %s' % (fn))
            pyplot.savefig(fn, metadata=fig_meta)
            if save_png:
                fn_png = fn.replace('.pdf', '.png')
                print('save figure to file: %s' % (fn_png))
                pyplot.savefig(fn_png, metadata=fig_meta)
            if copy2tex:
                fn_paper = fn.replace(plots_folder, paper_folder)
                cmd = 'cp %s %s' % (fn, fn_paper)
                print(cmd)
                os.system(cmd)

In [ ]:
sum((y_pred != 2)) + sum( (y_pred != 1))

In [ ]:
len(y_pred)

In [ ]:
cm

In [ ]:
8.3 * np.deg2rad(2.)